<a href="https://colab.research.google.com/github/VictorSantos117/Entrega-1-PI3/blob/main/1Entrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


library(httr2)

# A função abaixo baixa os artigos da wikipedia

baixar_wiki <- function(titulo) {

  request("https://pt.wikipedia.org/w/api.php") |>
    req_url_query(
      action = "query",
      prop = "extracts",
      explaintext = 1,
      format = "json",
      redirects = 1,
      titles = titulo
    ) |>
    req_perform() |>
    resp_body_json() |>
    (\(r) r$query$pages[[1]]$extract)()
}


municipios <- c(
  santos = "Bolsa Oficial de Café",
  praia_grande = "Fortaleza de Itaipu",
  sao_vicente = "São Vicente (São Paulo)"
)


docs <- sapply(municipios, baixar_wiki)



# Lista de stopwords para evitar que elas apareçam mais que palavras chave importantes

stopwords_pt <- c(
  "a", "à", "às", "ao", "aos", "as", "até",
  "com", "como", "da", "das", "de", "dela", "delas",
  "dele", "deles", "do", "dos", "e", "é", "em",
  "entre", "era", "eram", "essa", "essas", "esse",
  "esses", "esta", "estas", "este", "estes", "foi",
  "foram", "há", "isso", "isto", "já", "la", "lá",
  "lhe", "lhes", "mais", "mas", "me", "mesmo",
  "mesmos", "minha", "minhas", "muito", "muitos",
  "na", "nas", "não", "no", "nos", "nós", "nossa",
  "nossas", "nosso", "nossos", "num", "numa",
  "o", "os", "ou", "para", "pela", "pelas", "pelo",
  "pelos", "por", "qual", "quando", "que", "quem",
  "se", "sem", "ser", "seu", "seus", "só", "sob",
  "sobre", "sua", "suas", "também", "te", "tem",
  "têm", "tendo", "tenho", "ter", "teu", "teus",
  "tu", "um", "uma", "umas", "uns", "vai", "vão",
  "você", "vocês"
)


# Limpeza e tokenização


tokenizar_limpar <- function(texto) {


  texto <- tolower(texto)



  texto <- gsub(
    "https?://[^[:space:]]+",
    " ",
    texto
  )



  texto <- gsub(
    "[[:punct:]]+",
    " ",
    texto
  )


  texto <- gsub(
    "[0-9]+",
    " ",
    texto
  )



  tokens <- unlist(
    strsplit(texto, "\\s+")
  )



  tokens <- tokens[tokens != ""]



  tokens <- tokens[
    !tokens %in% stopwords_pt
  ]



  tokens <- tokens[
    nchar(tokens) >= 3
  ]


  return(tokens)
}


tokens <- lapply(
  docs,
  tokenizar_limpar
)




cat("Quantidade de tokens após a limpeza:\n")

for (i in seq_along(tokens)) {

  cat(
    " -", names(tokens)[i],
    ":", length(tokens[[i]]),
    "tokens\n"
  )
}


vocab <- sort(
  unique(
    unlist(tokens)
  )
)


cat("\n============================================\n")
cat("              VOCABULÁRIO\n")
cat("============================================\n\n")

cat(
  "Quantidade de termos diferentes:",
  length(vocab),
  "\n"
)



freq <- table(
  unlist(tokens)
)


top10 <- sort(
  freq,
  decreasing = TRUE
)[1:10]


cat("\n============================================\n")
cat("       10 TERMOS MAIS FREQUENTES\n")
cat("============================================\n\n")

print(top10)


cat("\nRanking dos termos:\n\n")

for (i in seq_along(top10)) {

  cat(
    i, "º -",
    names(top10)[i],
    ":",
    as.integer(top10[i]),
    "ocorrências\n"
  )
}


tdm <- sapply(
  tokens,
  function(tk) {

    as.integer(
      table(
        factor(
          tk,
          levels = vocab
        )
      )
    )
  }
)


rownames(tdm) <- vocab

cat("\n============================================\n")
cat("                 TF-IDF\n")
cat("============================================\n\n")



N <- ncol(tdm)


total_termos <- colSums(tdm)


tf <- sweep(
  tdm,
  2,
  total_termos,
  "/"
)

documentos_com_termo <- rowSums(tdm > 0)

idf <- log(
  N / documentos_com_termo
)

tfidf <- tf * idf


cat("Número de termos:", nrow(tfidf), "\n")
cat("Número de documentos:", ncol(tfidf), "\n")


for (documento in colnames(tfidf)) {

  cat("\n--------------------------------------------\n")
  cat("Termos mais importantes para:", documento, "\n")
  cat("--------------------------------------------\n\n")


  valores <- tfidf[, documento]


  top <- sort(
    valores,
    decreasing = TRUE
  )[1:10]


  for (i in seq_along(top)) {

    cat(
      i, "º -",
      names(top)[i],
      ":",
      round(top[i], 4),
      "\n"
    )
  }
}


cat("\n============================================\n")
cat("       MATRIZ TERMO-DOCUMENTO\n")
cat("============================================\n\n")

cat(
  "Número de termos:",
  nrow(tdm),
  "\n"
)

cat(
  "Número de documentos:",
  ncol(tdm),
  "\n"
)

cat(
  "Dimensão:",
  nrow(tdm),
  "x",
  ncol(tdm),
  "\n"
)



cat("\nPrimeiros 10 termos da matriz:\n\n")

print(
  tdm[1:min(10, nrow(tdm)), ]
)


busca_booleana <- function(termo, tdm) {

  termo <- tolower(termo)


  if (!termo %in% rownames(tdm)) {

    return(character(0))
  }


  colnames(tdm)[
    tdm[termo, ] > 0
  ]
}



Quantidade de tokens após a limpeza:
 - santos : 497 tokens
 - praia_grande : 305 tokens
 - sao_vicente : 1215 tokens

              VOCABULÁRIO

Quantidade de termos diferentes: 1205 

       10 TERMOS MAIS FREQUENTES


    são vicente    café  brasil    vila   paulo  santos  cidade   bolsa oficial 
     59      39      38      22      20      19      18      17      13      11 

Ranking dos termos:

1 º - são : 59 ocorrências
2 º - vicente : 39 ocorrências
3 º - café : 38 ocorrências
4 º - brasil : 22 ocorrências
5 º - vila : 20 ocorrências
6 º - paulo : 19 ocorrências
7 º - santos : 18 ocorrências
8 º - cidade : 17 ocorrências
9 º - bolsa : 13 ocorrências
10 º - oficial : 11 ocorrências

                 TF-IDF

Número de termos: 1205 
Número de documentos: 3 

--------------------------------------------
Termos mais importantes para: santos 
--------------------------------------------

1 º - café : 0.084 
2 º - bolsa : 0.0287 
3 º - museu : 0.0199 
4 º - palácio : 0.0111 
5 º - ca